# Editing-cycle simulation — full pipeline

Replaces the human reviewer with an AI judge:

1. **preedit** — read existing robot predictions, parse top-1 *(no GPU)*
2. **judge** — stronger model reviews from image *(Azure API, no GPU)*
3. **postedit** — robot re-diagnoses given feedback *(GPU)*
4. **eval** — pre/judge/post accuracy vs MIDAS ground truth (y16)

Run on a **GPU node** (phase 3 needs it). All outputs go to `prelim_simedit/results_local/`.

In [1]:
import os, sys

PROJECT_ROOT = "/scratch/jq2uw/derm_vlms"
SIMEDIT = os.path.join(PROJECT_ROOT, "prelim_simedit")
if SIMEDIT not in sys.path:
    sys.path.insert(0, SIMEDIT)

from utils.pipeline import run_preedit, run_judge, run_postedit, run_eval
from utils.io import set_output_dir

# --- Configuration ---
ROBOT = "dermato_llama"         # candidates: medgemma, dermato_llama
JUDGE = "ground_truth"            # judges: gpt53, gpt54, claude_opus48, claude_opus46, claude_sonnet46, claude_fable, ground_truth
SEED = 42                  # for reproducibility (used wherever randomness is needed)
DIFFERENTIAL = "top_3"     # "top_1" = single diagnosis, "top_3" = top-3 differential

# Filtering (pick ONE approach):
N = 5                      # first N cases (set None for ALL)
CASE_IDS = None            # OR explicit list: ["1_combined", "5_combined", "10_combined"]

# Notebook test outputs go to results_local/test/ (can overwrite freely).
# Set to None to write to results_local/<robot>/ (official runs).
set_output_dir(os.path.join(SIMEDIT, "results_local", "test"))

In [2]:
# --- Phase 1: preedit (no GPU) ---
# Reads existing predictions CSV, parses diagnosis.
# Takes first N cases (sorted by case_id). Set CASE_IDS to override.

df_preedit = run_preedit(ROBOT, n=N, seed=SEED, case_ids=CASE_IDS,
                         differential=DIFFERENTIAL)
df_preedit[["case_id", "gt_y16", "preedit_dx"]]

build_inputs(dermato_llama): 5 combined cases
[preedit/dermato_llama/top_3] 5 cases -> /scratch/jq2uw/derm_vlms/prelim_simedit/results_local/test/01_preedit__top_3.csv


,case_id,gt_y16,preedit_dx
0,1_combined,Squamous Cell Carcinoma In Situ,1. Basal Cell Carcinoma: The lesion displays a...
1,2_combined,Melanocytic Nevus,1. **Nevus**: The lesion appears to be a beni...
2,6_combined,Squamous Cell Carcinoma,1. Squamous Cell Carcinoma: The lesion exhibit...
3,8_combined,Other,1. **Seborrheic Keratosis**: The lesion presen...
4,9_combined,Seborrheic Keratosis,1. Squamous Cell Carcinoma: The lesion exhibit...


In [3]:
# --- Phase 2: judge (API, no GPU) ---
# Judge sees the image + robot's diagnosis, provides its own.
# Resumable: skips case_ids already in the judge CSV.

df_judge = run_judge(robot_name=ROBOT, judge=JUDGE, differential=DIFFERENTIAL)
df_judge[[c for c in df_judge.columns if c.startswith("judge_") or c in ("case_id", "gt_y16")]]

[judge/dermato_llama/ground_truth/top_3] 0/5 pending


,case_id,gt_y16,judge_corrected_differential
0,1_combined,Squamous Cell Carcinoma In Situ,1. Squamous Cell Carcinoma In Situ: Ground tru...
1,2_combined,Melanocytic Nevus,1. Melanocytic Nevus: Ground truth diagnosis
2,6_combined,Squamous Cell Carcinoma,1. Squamous Cell Carcinoma: Ground truth diagn...
3,8_combined,Other,1. Other: Ground truth diagnosis
4,9_combined,Seborrheic Keratosis,1. Seborrheic Keratosis: Ground truth diagnosis


In [4]:
# --- Phase 3: postedit (GPU) ---
# Robot re-diagnoses given the judge's feedback.
# Resumable: skips case_ids already in the postedit CSV.

df_postedit = run_postedit(ROBOT, judge_name=JUDGE, differential=DIFFERENTIAL)
if DIFFERENTIAL == "top_1":
    df_postedit[["case_id", "gt_y16", "preedit_dx", "judge_dx", "postedit_dx"]]
else:
    df_postedit[["case_id", "gt_y16", "postedit_dx"]]

[postedit/dermato_llama/ground_truth/top_3] 0/5 pending


In [5]:
# --- Phase 4: eval (no GPU) ---
# Score all phases against ground truth (y16).

scored, summary = run_eval(robot_name=ROBOT, judge_name=JUDGE,
                           differential=DIFFERENTIAL)

[eval/dermato_llama/ground_truth/top_3] -> /scratch/jq2uw/derm_vlms/prelim_simedit/results_local/test/scored__ground_truth__top_3.csv
        robot        judge differential  n  preedit_top1  preedit_top3  judge_top1  judge_top3  postedit_top1  postedit_top3  delta_top1  delta_top3  n_improved  n_regressed
dermato_llama ground_truth        top_3  5           0.2           0.2         1.0         1.0            0.4            0.4         0.2         0.2           1            0


In [6]:
# --- Per-case details ---
if DIFFERENTIAL == "top_1":
    df = scored[["case_id", "gt_y16",
            "preedit_dx", "preedit_y16", "preedit_correct",
            "judge_dx", "judge_y16", "judge_correct",
            "postedit_dx", "postedit_y16", "postedit_correct"]]
else:
    df = scored[["case_id", "gt_y16",
            "preedit_dx1", "preedit_top1_correct", "preedit_top3_correct",
            "judge_dx1", "judge_top1_correct", "judge_top3_correct",
            "postedit_dx1", "postedit_top1_correct", "postedit_top3_correct"]]

df

,case_id,gt_y16,preedit_dx1,preedit_top1_correct,preedit_top3_correct,judge_dx1,judge_top1_correct,judge_top3_correct,postedit_dx1,postedit_top1_correct,postedit_top3_correct
0,1_combined,Squamous Cell Carcinoma In Situ,Basal Cell Carcinoma,False,False,Squamous Cell Carcinoma In Situ,True,True,non-invasive form of squamous cell carcinoma,False,False
1,2_combined,Melanocytic Nevus,Nevus,True,True,Melanocytic Nevus,True,True,melanocytic nevus,True,True
2,6_combined,Squamous Cell Carcinoma,actinic keratosis,False,False,Squamous Cell Carcinoma,True,True,squamous cell carcinoma,True,True
3,8_combined,Other,actinic keratosis,False,False,Other,True,True,seborrheic keratosis,False,False
4,9_combined,Seborrheic Keratosis,actinic keratosis,False,False,Seborrheic Keratosis,True,True,this diagnosis,False,False
